# 🎬 AutoDub Studio 2.0 — Chatterbox-Only Colab Deployment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedj7895-cell/ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-ElevenLabs-Quality-Open-Source-/blob/main/AutoDub%20Studio%202.0.ipynb)

ElevenLabs-quality **open-source automatic dubbing** — Demucs v4 · Pyannote 3.1 ·
SenseVoice-Small · **Chatterbox multilingual zero-shot cloning** — wrapped in
an iPhone-15 frosted-glass UI.

> **This is the Chatterbox-only build.** CosyVoice 2/3, Fish S2-Pro, Kokoro-82M
> and Edge-TTS are removed here: Tab 1's engine selector offers Chatterbox and
> nothing else, and Chatterbox is the default. The pipeline, UI and three tabs
> are otherwise identical to `Colab_Runner.ipynb`, which keeps all six engines.

**How to use**
1. ▸ *Runtime* ▸ *Change runtime type* ▸ **T4 GPU** ▸ Save
2. ▸ *Runtime* ▸ **Run all**
3. When the last cell finishes, click the printed **`https://….gradio.live`** link
4. In Tab 1: upload your **audio/video master + Original SRT + Translated SRT**, hit 🔍
5. Tab 2: 🧬 matches speakers & emotions (paste a HuggingFace token in Tab 1 ▸ Advanced first — see [token guide](https://github.com/syedj7895-cell/ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-ElevenLabs-Quality-Open-Source-#-one-time-hugging-face-setup-required-for-step-3))
6. Tab 3: 🚀 renders the dubbed master mix / video

Ready to use — no build step required.

In [ ]:
# ── 0 ▸ 🩺 DIAGNOSTICS — run me FIRST (and whenever anything breaks) ──
# Prints one copy-paste block: versions · GPU · disk/RAM · HF gate access.
import importlib.metadata as md, os, shutil, sys, urllib.request

print("═" * 64)
print(" 🩺 AUTODUB DIAGNOSTICS — paste this whole block with any bug report")
print("═" * 64)
print(f"python          {sys.version.split()[0]}")
print(f"platform        {sys.platform}")

_pkgs = ["torch", "torchaudio", "torchvision", "numpy", "scipy", "librosa",
         "soundfile", "gradio", "gradio-client", "huggingface-hub",
         "demucs", "pyannote-audio", "funasr", "modelscope", "pysrt",
         "moviepy", "pydub", "ffmpeg-python", "setuptools"]
for name in _pkgs:
    try:
        print(f"{name:16s} {md.version(name)}")
    except Exception:
        print(f"{name:16s} —")

try:
    import torch
    if torch.cuda.is_available():
        pr = torch.cuda.get_device_properties(0)
        print(f"gpu              {pr.name} · {pr.total_memory / 2**30:.0f} GB · CUDA {torch.version.cuda}")
    else:
        print("gpu              NONE — CPU mode (expected during Colab GPU cooldown)")
except Exception as e:
    print("gpu              probe failed:", e)

print(f"disk  /content   {shutil.disk_usage('/content').free / 2**30:.0f} GB free")
try:
    _m = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES')
    print(f"ram              {_m / 2**30:.0f} GB")
except Exception:
    pass
print("ffmpeg           " + (shutil.which("ffmpeg") or "NOT FOUND"))

# ── HF gate probe (needs a token to be meaningful) ────────────────────
_token = (os.environ.get("HF_TOKEN") or "").strip()
if not _token and os.path.exists("/content/autodub_hf_token.txt"):
    _token = open("/content/autodub_hf_token.txt").read().strip()
if not _token:
    print("hf-gate          ⚠ no token found — paste it in Tab 1 ▸ Advanced, "
          "or set HF_TOKEN, then re-run this cell")
else:
    for repo in ("pyannote/speaker-diarization-3.1",
                 "pyannote/segmentation-3.0"):
        req = urllib.request.Request(
            f"https://huggingface.co/api/models/{repo}",
            headers={"Authorization": f"Bearer {_token}"})
        try:
            with urllib.request.urlopen(req, timeout=15) as r:
                print(f"hf-gate          ✅ {repo} accessible")
        except Exception as e:
            code = getattr(e, "code", None)
            if code in (401, 403):
                print(f"hf-gate          ❌ {repo} DENIED — open huggingface.co/{repo} "
                      "and click 'Agree and access repository' with the SAME account "
                      "that owns the token")
            else:
                print(f"hf-gate          ⚠ {repo} probe failed ({e}) — check network")
print("═" * 64)

In [ ]:
# -- 1 > Runtime check + PINNED ML-stack install (fail-soft, torch untouched) --
import importlib.metadata as md, subprocess, sys, torch

if torch.cuda.is_available():
    print(f"[OK] GPU ready: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB)")
else:
    print("[!] No GPU on this VM - CPU mode (fine for Tabs 1-2 testing; "
          "full renders want the T4. Runtime > Change runtime type > T4 GPU).")

# Exact known-good pins - PROVEN to install on Colab py3.13 (verified live).
# torch / torchaudio / numpy / soundfile are deliberately NOT here: Colab's
# own builds must stay (GPU-CUDA wheel + system compatibility).
PINNED = {
    "demucs":          "4.0.1",
    "pyannote.audio":  "3.1.1",
    "funasr":          "1.2.6",
    "modelscope":      "1.27.1",
    "pysrt":           "1.1.2",
    "pydub":           "0.25.1",
    "moviepy":         "1.0.3",
    "ffmpeg-python":   "0.2.0",
    "librosa":         "0.11.0",
}
need = []
for pkg, ver in PINNED.items():
    try:
        have = md.version(pkg)
        ok = have == ver
    except Exception:
        have, ok = None, False
    status = ("[OK] " + str(have)) if ok else (str(have or "missing")
             + " -> will install " + ver)
    print(f"  {pkg:18s} {status}")
    if not ok:
        need.append(f"{pkg}=={ver}")

if need:
    print(f"\nInstalling {len(need)} pinned package(s) ...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", *need],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print("install done")
    else:
        # FAIL-SOFT: show the REAL pip error, keep 'Run all' going
        print("[!] pip INSTALL FAILED - tail of pip output:")
        for ln in (r.stdout or "").splitlines()[-25:]:
            print("   ", ln)
        for ln in (r.stderr or "").splitlines()[-10:]:
            print("   ", ln)
        print("[!] WARNING: ML stack incomplete - later steps may fail. "
              "Fix the error above and re-run this cell.")

# demucs.api (Separator) is MISSING from the PyPI 4.0.1 wheel - it only
# exists in the GitHub source. Install demucs from source when absent.
import importlib.util
if importlib.util.find_spec("demucs.api") is None:
    print("demucs.api missing from the PyPI wheel - installing demucs from source ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "--no-deps",
                           "git+https://github.com/facebookresearch/demucs.git"])
    print("demucs source install done")

# verification imports - fail-soft (never halt 'Run all')
try:
    import demucs, pyannote, pysrt, funasr, modelscope  # noqa: F401
    from demucs.api import Separator  # noqa: F401
    print("[OK] ML stack verified: demucs / demucs.api / pyannote / funasr / modelscope / pysrt")
except Exception as e:
    print(f"[!] WARNING: stack verification failed ({e}) - "
          "re-run this cell after fixing the install error above.")

In [ ]:
# ── 2 ▸ Clone the project ─────────────────────────────────────────────
REPO_URL = ("https://github.com/syedj7895-cell/"
            "ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-"
            "ElevenLabs-Quality-Open-Source-.git")
!git clone {REPO_URL} ai-video-dubber
%cd ai-video-dubber
!ls

In [ ]:
# ── realign gradio-client to gradio's EXACT pin + full-stack audit ──
import importlib.metadata as md, os, subprocess, sys
try:
    g = md.version("gradio")
    want = next((r.split("==")[1].strip() for r in md.requires("gradio")
                 if r.startswith("gradio-client")), None)
    have = md.version("gradio_client")
    if want and have != want:
        print(f"gradio {g}: gradio-client {have} -> {want}")
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "-q", f"gradio-client=={want}"])
    else:
        print(f"gradio {g} + gradio-client {have} already aligned")
except Exception as e:
    print("client realignment skipped:", e)

# audit: every remaining unpinned requirement from the project's file
_req = os.path.join(os.getcwd(), "requirements.txt")   # we are inside the clone
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", _req])
print("✅ requirements.txt satisfied")


In [ ]:
# -- 4b > CACHE CHECK / RESTORE (fast; run before the model pre-download) --
# If a prior model cache exists, reuse it so this session skips the
# multi-GB downloads. Priority: HF Hub (if token below) -> Google Drive.
# If neither is available, nothing happens and models download normally.
#
# IMPORTANT: the cache repo is PRIVATE, so restore MUST authenticate.
# Paste the SAME token you used to UPLOAD. An unauthenticated download
# returns 401, which looks exactly like "empty storage".
HF_TOKEN_RESTORE = ""   # optional: paste your HF token to restore from HF Hub
HF_REPO_RESTORE  = ""   # optional: e.g. "yourname/autodub-model-cache"

import os as _os, pathlib as _pl, shutil as _sh, json as _json
_HOME = _os.path.expanduser("~")
_ms_cache = _pl.Path(_os.environ.get("MODELSCOPE_CACHE", _HOME + "/.cache/modelscope"))
_hf_cache = _pl.Path(_os.environ.get("HF_HOME", _HOME + "/.cache/huggingface"))
_ms_cache.mkdir(parents=True, exist_ok=True)
_hf_cache.mkdir(parents=True, exist_ok=True)
_os.environ.setdefault("MODELSCOPE_CACHE", str(_ms_cache))
_os.environ.setdefault("HF_HOME", str(_hf_cache))

def _copy_tree(src, dst):
    n = 0
    for f in _pl.Path(src).rglob("*"):
        if f.is_file() and f.suffix != ".lock" and "tmp" not in f.name:
            rel = f.relative_to(src)
            out = _pl.Path(dst) / rel
            out.parent.mkdir(parents=True, exist_ok=True)
            if not out.exists():
                _sh.copy2(str(f), str(out))
                n += 1
    return n

_restored = False
_stage = _pl.Path("/content/autodub_restore")

# -- Priority 1: HF Hub (needs a token) -------------------------------
# Token resolution order: pasted here -> Colab Secrets / env var ->
# the token file other cells use. So once HF_TOKEN exists in the runtime
# this cell needs no editing at all.
_tok = (HF_TOKEN_RESTORE or "").strip()
if not _tok:
    _tok = (_os.environ.get("HF_TOKEN")
            or _os.environ.get("HUGGING_FACE_HUB_TOKEN") or "").strip()
    if _tok:
        print("Cache restore: using HF_TOKEN from the environment")
if not _tok and (_pl.Path("/content/autodub_hf_token.txt").exists()):
    _tok = _pl.Path("/content/autodub_hf_token.txt").read_text().strip()
    if _tok:
        print("Cache restore: using /content/autodub_hf_token.txt")
if _tok:
    # Make the token AMBIENT so every hub call this session (pyannote,
    # Chatterbox, snapshot_download) is authenticated by default.
    _os.environ["HF_TOKEN"] = _tok
    _os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
    try:
        from huggingface_hub import HfApi, snapshot_download
        # Best-effort global login. It must NEVER abort the restore: even if
        # it fails, HfApi(token=) + snapshot_download(token=) still work.
        try:
            from huggingface_hub import login
            try:
                login(token=_tok, add_to_git_credential=False)
            except TypeError:
                login(_tok)
        except Exception as _le:
            print("Cache restore: hub login skipped (", str(_le)[:70], ")")
        _api = HfApi(token=_tok)

        # repo id: explicit override -> saved config -> convention
        try:
            _cfg = _json.loads(_pl.Path("/content/autodub_persist.json").read_text())
        except Exception:
            _cfg = {}
        _cands, _who = [], None
        try:
            _who = _api.whoami().get("name", "user")
            print("Cache restore: token OK as", _who)
        except Exception as e:
            print("Cache restore: TOKEN REJECTED (", str(e)[:90], ")")
            print("  -> regenerate it at huggingface.co/settings/tokens")
        if _who:
            for _c in (HF_REPO_RESTORE, _cfg.get("hf_repo"),
                       _who + "/autodub-model-cache"):
                _c = (_c or "").strip()
                if _c and _c not in _cands:
                    _cands.append(_c)
            for _repo in _cands:
                try:
                    _files = _api.list_repo_files(repo_id=_repo,
                                                  repo_type="dataset", token=_tok)
                except Exception as e:
                    _m = str(e)
                    if "401" in _m or "403" in _m or "Unauthorized" in _m:
                        print("Cache restore:", _repo, "-> token has NO ACCESS (401)")
                    elif "404" in _m or "not found" in _m.lower():
                        print("Cache restore:", _repo, "-> no such private dataset")
                    else:
                        print("Cache restore:", _repo, "->", _m[:90])
                    continue
                if not _files:
                    print("Cache restore:", _repo, "-> accessible but EMPTY (0 files)")
                    continue
                _sh.rmtree(_stage, ignore_errors=True)
                # token= is REQUIRED here: the repo is private
                snapshot_download(_repo, repo_type="dataset", token=_tok,
                                  local_dir=str(_stage))
                a = _copy_tree(_stage / "modelscope", _ms_cache)
                b = _copy_tree(_stage / "hf", _hf_cache)
                if not (a + b):
                    a = _copy_tree(_stage, _ms_cache)
                print("Cache restore: HF Hub OK -", _repo,
                      "(" + str(a + b) + " file(s) restored,",
                      str(len(_files)) + " in repo)")
                _restored = True
                break
            if not _restored:
                print("Cache restore: no usable HF cache in", _cands)
    except Exception as e:
        print("Cache restore: HF step skipped (", str(e)[:120], ")")
else:
    print("Cache restore: no HF token (paste one above, or set HF_TOKEN) -> trying Google Drive next")

# -- Priority 2: Google Drive -----------------------------------------
if not _restored:
    try:
        from google.colab import drive as _drive
        _drv = _pl.Path("/content/drive/MyDrive/AutoDub_Studio/model_cache")
        if not _drv.exists():
            _md = _pl.Path("/content/drive/MyDrive")
            if not _md.exists():
                print("Cache restore: mounting Google Drive (one permission click) ...")
                _drive.mount("/content/drive")
        if _drv.exists():
            a = _copy_tree(_drv / "modelscope", _ms_cache)
            b = _copy_tree(_drv / "hf", _hf_cache)
            print("Cache restore: Drive OK (" + str(a + b) + " file(s) restored)")
            _restored = True
        else:
            print("Cache restore: no Drive cache found")
    except Exception as e:
        print("Cache restore: Drive step skipped (", str(e)[:120], ")")

if not _restored:
    print("Cache restore: no prior cache -> models will download normally.")
else:
    print("Cache restore complete - the next cell will reuse these models.")


In [ ]:
# ── 5 ▲ Persistence + PRE-DOWNLOAD (Chatterbox only) ─────────────
# AutoDub Studio 2.0: the CosyVoice bootstrap that used to sit here is gone.
# Chatterbox installs itself on demand inside pipeline.py (--no-deps), so no
# source tree and no pinned dependency set are required up front.
import json, os, pathlib, re, site, subprocess, sys, time

# 6 - OPTIONAL: persist outputs/ + uploads/ (+ model caches) to Google Drive
#     so a Colab disconnect never loses pipeline progress. Set False to skip.
_persist = {"mode": "none", "hf_token": "", "auto_cleanup": True}
_pcfg = pathlib.Path("/content/autodub_persist.json")
if _pcfg.exists():
    try:
        _persist.update(json.loads(_pcfg.read_text(encoding="utf-8")))
    except Exception:
        pass
PERSIST_TO_DRIVE = (_persist.get("mode") == "drive")
import shutil as _sh
if PERSIST_TO_DRIVE:
    try:
        from google.colab import drive as _drive
        if not os.path.exists("/content/drive/MyDrive"):
            _drive.mount("/content/drive")
        _drv = pathlib.Path("/content/drive/MyDrive/AutoDub_Studio")
        _drv.mkdir(parents=True, exist_ok=True)
        _proj = pathlib.Path.cwd()          # the clone dir (Cell 3 cd'd here)
        for _name in ("outputs", "uploads"):
            _src, _dst = _proj / _name, _drv / _name
            _dst.mkdir(parents=True, exist_ok=True)
            if _src.exists() and not _src.is_symlink():
                for _f in _src.iterdir():
                    if not (_dst / _f.name).exists():
                        _sh.move(str(_f), str(_dst / _f.name))
                _sh.rmtree(_src, ignore_errors=True)
            if not _src.is_symlink():
                os.symlink(str(_dst), str(_src))
        if True:  # auto-cleanup: cloud stores models only
            _mc = _drv / "model_cache"
            _mc.mkdir(exist_ok=True)
            os.environ.setdefault("MODELSCOPE_CACHE", str(_mc / "modelscope"))
            os.environ.setdefault("HF_HOME", str(_mc / "huggingface"))
        print("OK outputs/ + uploads/ persist to Google Drive "
              "(MyDrive/AutoDub_Studio) - disconnects no longer lose progress"
              + (" + model caches" if PERSIST_MODEL_CACHE else ""))
    except Exception as e:
        print("[!] Drive persistence skipped:", str(e)[:160])

# 6b - HF-Hub persistence backend (models only; job files stay local)
if _persist.get("mode") == "hf" and _persist.get("hf_token"):
    import shutil as _sh
    try:
        from huggingface_hub import HfApi
        _api = HfApi(token=_persist["hf_token"])
        _user = _api.whoami().get("name", "user")
        _repo = f"{_user}/autodub-model-cache"
        _persist["hf_repo"] = _repo
        _pcfg = pathlib.Path("/content/autodub_persist.json")
        _pcfg.write_text(json.dumps(_persist), encoding="utf-8")
        try:
            _api.create_repo(_repo, repo_type="dataset", private=True,
                             exist_ok=True)
        except Exception:
            pass
        try:
            # token= is REQUIRED: _repo is PRIVATE. Without it this pull
            # 401s and silently looks like an empty cache.
            os.environ["HF_TOKEN"] = _persist["hf_token"]
            from huggingface_hub import snapshot_download as _hfsnap
            _hfsnap(_repo, repo_type="dataset",
                    token=_persist["hf_token"],
                    local_dir="/root/.cache/autodub_pull")
            _src = pathlib.Path("/root/.cache/autodub_pull")
            _n_pulled = sum(1 for _f in _src.rglob("*") if _f.is_file())
            if _n_pulled == 0:
                print("[i] HF cache pull: repo accessible but EMPTY")
            _src = pathlib.Path("/root/.cache/autodub_pull")
            if _src.exists():
                for _f in _src.rglob("*"):
                    if _f.is_file():
                        _rel = _f.relative_to(_src)
                        _dst = pathlib.Path("/root/.cache") / _rel
                        _dst.parent.mkdir(parents=True, exist_ok=True)
                        if not _dst.exists():
                            _sh.copy2(str(_f), str(_dst))
                print("OK model cache restored from HF Hub")
        except Exception as e:
            print("[i] HF cache pull skipped:", str(e)[:120])
    except Exception as e:
        print("[!] HF persistence setup failed:", str(e)[:160])

# 7 - PRE-DOWNLOAD all model checkpoints NOW (before GPU work) so Tabs 2-3
#     never stall on slow downloads while the GPU sits idle.
print("Pre-downloading model checkpoints (one-time, then cached) ...")
try:
    from modelscope import snapshot_download as _msdl
    for _repo in ("iic/SenseVoiceSmall",):
        try:
            _msdl(_repo)
            print(f"  cached: {_repo}")
        except Exception as e:
            print(f"  [!] {_repo}: {str(e)[:120]}")
except Exception as e:
    print("  [!] modelscope pre-download skipped:", str(e)[:120])

# Chatterbox - this build's ONLY TTS engine. Public repo, no token needed.
# allow_patterns mirrors pipeline._chatterbox_ckpt_dir exactly, so the first
# render never sits on a multi-GB download while the GPU idles.
try:
    from huggingface_hub import snapshot_download as _cbsdl
    _cb = _cbsdl("ResembleAI/chatterbox",
                 allow_patterns=["ve.pt", "t3_mtl23ls_v2.safetensors",
                                 "s3gen.pt", "conds.pt",
                                 "grapheme_mtl_merged_expanded_v1.json",
                                 "Cangjie5_TC.json"])
    print("  cached: ResembleAI/chatterbox ->", _cb)
except Exception as e:
    print(f"  [!] ResembleAI/chatterbox: {str(e)[:120]}")

_tok = (os.environ.get("HF_TOKEN") or "").strip()
if not _tok and os.path.exists("/content/autodub_hf_token.txt"):
    _tok = open("/content/autodub_hf_token.txt").read().strip()
if _tok:
    try:
        from huggingface_hub import snapshot_download as _hsdl
        for _repo in ("pyannote/speaker-diarization-3.1",
                      "pyannote/segmentation-3.0"):
            try:
                _hsdl(_repo, token=_tok)
                print(f"  cached: {_repo}")
            except Exception as e:
                print(f"  [!] {_repo}: {str(e)[:120]}")
    except Exception as e:
        print("  [!] pyannote pre-download skipped:", str(e)[:120])
else:
    print("  (i) no HF token found yet - pyannote will download during Tab 2")


# 8 - push model caches to HF Hub (hf mode only; models-only auto-cleanup)
if _persist.get("mode") == "hf" and _persist.get("hf_token") \
        and _persist.get("hf_repo"):
    import shutil as _sh
    try:
        from huggingface_hub import HfApi
        _api = HfApi(token=_persist["hf_token"])
        _stage = pathlib.Path("/root/.cache/autodub_stage")
        _sh.rmtree(_stage, ignore_errors=True)
        for _pref, _src in (("modelscope", "/root/.cache/modelscope"),
                            ("hf", "/root/.cache/huggingface")):
            _s = pathlib.Path(_src)
            if _s.exists():
                (_stage / _pref).mkdir(parents=True, exist_ok=True)
                for _f in _s.rglob("*"):
                    if _f.is_file() and _f.suffix != ".lock":
                        _rel = _f.relative_to(_s)
                        _d = _stage / _pref / _rel
                        _d.parent.mkdir(parents=True, exist_ok=True)
                        if not _d.exists():
                            _sh.copy2(str(_f), str(_d))
        _api.upload_folder(folder_path=str(_stage),
                           repo_id=_persist["hf_repo"],
                           repo_type="dataset",
                           commit_message="AutoDub model cache sync")
        print("OK model cache pushed to HF Hub:", _persist["hf_repo"])
    except Exception as e:
        print("[!] HF cache push skipped:", str(e)[:160])


In [ ]:
# ── 6 ▲ 🚀 LAUNCH - click the https://….gradio.live link printed below ─
# AutoDub Studio 2.0 is Chatterbox-only: the line below publishes NOTHING else
# in Tab 1's engine selector and makes Chatterbox the default engine. Delete it
# to get the full six-engine registry back - that is all Colab_Runner.ipynb does.
import os
os.environ["AUTODUB_TTS_ENGINES"] = "chatterbox"
!python app.py